In [1]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


# Read the Target Sequence / Profile

In [2]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

# Parse the sequences 

In [13]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

# Initialize HMM

In [4]:
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

# Convert to HQMM

In [14]:
from modules.qmodel import (
    build_hqmm,
    convert_to_torch
)

K = build_hqmm(trans_matdf, emit_matdf)
K_torch = convert_to_torch(K)

# Train HQMM

In [6]:
import torch
import numpy as np
from pathlib import Path

from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm

# Directory for saved models
model_dir = Path("saved_models/phqmm")
model_dir.mkdir(parents=True, exist_ok=True)

# Initial density matrix
p = torch.zeros(K[0, 1].shape, dtype=torch.complex64)
p[0, 0] = 1.0

W = match_len * 2 + 1

da0 = description_accuracy_hqmm(
    p, K_torch, W, sym_idx, test_seqs
)

results = {
    "untrained": (float(da0.mean()), float(da0.std()))
}
histories = {}
trained_models = {}

settings = {
    "large step (0.1)": 0.1,
    "medium step (0.01)": 0.01,
    "small step (0.001)": 0.001,
}

max_iter = 15
tol = 1e-3
batch_size = 16
alpha = 0.8

for label, tau in settings.items():
    print(f"--- {label}: tau={tau}", flush=True)

    K_initial = {
        key: value.detach().clone()
        for key, value in K_torch.items()
    }

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    # Save training results
    histories[label] = hist
    trained_models[label] = K_trained

    # Save checkpoint immediately after training
    model_path = model_dir / f"hqmm_tau_{tau:g}.pt"
    history_path = model_dir / f"history_tau_{tau:g}.csv"

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "W": W,
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # Align devices before evaluation
    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval, K_eval, W, sym_idx, test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Test DA: {results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

--- large step (0.1): tau=0.1
Training on cuda (dtype=torch.complex64)


KeyboardInterrupt: 

# Visualization

In [ ]:
import importlib
import modules.qplots as qplots

importlib.reload(qplots)
plot_da_bars = qplots.plot_da_bars

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    pass

# Create directory for saved models
results_dir = Path("results/phqmm")

# Save training histories
for label, h in histories.items():
    h.to_csv(
        f"q_history_{label.split()[0]}.csv",
        index=False
    )

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_q_training_histories.csv",
    index=False,
)

# Save final description accuracy results
pd.DataFrame(
    results,
    index=["test_da_mean", "test_da_std"]
).T.to_csv(results_dir / f"{name}_q_da_results.csv")

# Optional reference DA values
references = {}

# Test DA vs GPU training time
plot_da_learning_curve(
    histories,
    references,
    x="train_time",
    out=results_dir / f"{name}_q_da_vs_time.png"
)

# Test DA vs epoch
plot_da_learning_curve(
    histories,
    references,
    x="iter",
    out=results_dir / f"{name}_q_da_vs_epoch.png"
)

# Final DA comparison by step size
plot_da_bars(
    results,
    references,
    out=results_dir / f"{name}_q_da_results.png"
)